# PyADM1ODE benchmark: plug in a (free) LLM and score it

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/dgaida/PyADM1ODE/blob/master/benchmark/llm_benchmark_en.ipynb)

This notebook shows the complete benchmark run for **one** data point:

1. **Load the task** – a described biogas plant from the benchmark dataset.
2. **The LLM builds the model** – a freely chosen LLM generates PyADM1ODE code.
3. **Oracle** – for under-specified tasks the oracle answers the LLM's follow-up questions.
4. **Scoring** – the generated code is executed and scored against the reference
   (structure, measures, missing values).

The LLM is wired in through [**litellm**](https://github.com/BerriAI/litellm): a single
interface to 100+ providers. Switching provider is therefore a one-liner (just change
the `MODEL` variable).

More background: [Using the Dataset](https://dgaida.github.io/PyADM1ODE/en/benchmark/nutzung/)
and [How does the Oracle work?](https://dgaida.github.io/PyADM1ODE/en/benchmark/oracle/)

## 1. Setup

Install PyADM1ODE and litellm. **In Colab** run this cell. **Locally** (already inside
the repo) you can skip the `git clone`/`%cd` part.

In [ ]:
!git clone https://github.com/dgaida/PyADM1ODE.git
%cd PyADM1ODE
!pip install -q -e .
!pip install -q litellm

## 2. Provider & API key (free LLM)

By default this notebook uses the **free Groq API** (generous free tier). Get a key at
<https://console.groq.com/keys>.

litellm reads the key from the `GROQ_API_KEY` environment variable. For other
providers see section 7, which lists the variable each one expects.

In [ ]:
import os, getpass

# Freely chosen model in litellm format "<provider>/<model>".
MODEL = "groq/llama-3.3-70b-versatile"

if not os.environ.get("GROQ_API_KEY"):
    os.environ["GROQ_API_KEY"] = getpass.getpass("GROQ_API_KEY: ")

## 3. Load the harness and pick a data point

We use the real benchmark building blocks from `benchmark/eval/`: `build_messages`
(prompt), `extract_code`/`extract_questions` (parse the answer), `Oracle` (answer
follow-up questions) and `evaluate_code` (execute + score).

In [ ]:
import sys, json
from pathlib import Path

REPO = Path.cwd()
sys.path.insert(0, str(REPO / "benchmark" / "eval"))

from prompt import SYSTEM_PROMPT, build_messages, add_oracle_answers, append_assistant
from solve import extract_code, extract_questions
from oracle import Oracle
from runner import evaluate_code

DATASET = REPO / "benchmark" / "dataset"
index = json.loads((DATASET / "index.json").read_text(encoding="utf-8"))

# Tip: list all available data points for an overview
for e in index["datapoints"]:
    print(f"{e['id']:<22} {e['regime']:<16} {e['modality']:<7} {e['language']}")

In [ ]:
# Pick a data point.
# 'BGA1_terse_de' is under-specified -> activates the oracle.
DP_ID = "BGA1_terse_de"

entry = next(e for e in index["datapoints"] if e["id"] == DP_ID)
dp_path = DATASET / entry["path"]
dp = json.loads(dp_path.read_text(encoding="utf-8"))
dp_dir = str(dp_path.parent)

print("ID:      ", DP_ID)
print("Regime:  ", dp["regime"])
print("Modality:", dp["input"]["modality"], "| language:", dp["input"]["language"])
print("\nDescription:\n", dp["input"].get("content", "(image only)"))

## 4. Wiring the LLM through litellm

A single `call_llm` function encapsulates the provider. The system prompt (the full
PyADM1ODE API quick reference) is prepended. Text content is collapsed into a string
(maximum provider compatibility). Image content stays as an OpenAI vision list, so
vision models can read sketch data points.

In [ ]:
import litellm

def _to_litellm(messages):
    """prompt.py returns content as a list of parts. Collapse pure-text content into a
    string. Keep image_url parts unchanged (vision)."""
    norm = []
    for m in messages:
        c = m["content"]
        if isinstance(c, list) and all(p.get("type") == "text" for p in c):
            c = "\n".join(p["text"] for p in c)
        norm.append({"role": m["role"], "content": c})
    return norm

def call_llm(messages, model=MODEL, max_tokens=4096):
    full = [{"role": "system", "content": SYSTEM_PROMPT}] + _to_litellm(messages)
    resp = litellm.completion(model=model, messages=full, max_tokens=max_tokens, temperature=0.0)
    return resp.choices[0].message.content

## 5. Workflow: round 1 → oracle → round 2

The same logic as `benchmark/eval/solve.py`: for under-specified tasks the LLM may
ask follow-up questions first. If it does (instead of producing code immediately), the
oracle answers them and the LLM writes the finished code in round 2.

In [ ]:
regime = dp.get("regime", "underspecified")
allow_questions = regime == "underspecified"

messages = build_messages(dp, dp_dir, allow_questions=allow_questions)
resp1 = call_llm(messages)
print("=== Round 1 (LLM) ===\n", resp1[:1000])

code_str = extract_code(resp1)
questions = extract_questions(resp1)
oracle_turns = 0

if code_str is None and questions and allow_questions:
    oracle_turns = 1
    answer_text = Oracle(dp).answer(questions)
    print("\n=== Oracle answers ===\n", answer_text)
    append_assistant(messages, resp1)
    add_oracle_answers(messages, answer_text)
    resp2 = call_llm(messages)
    print("\n=== Round 2 (LLM) ===\n", resp2[:1000])
    code_str = extract_code(resp2)

assert code_str, "No Python code found in the LLM answer."
print("\nOracle rounds:", oracle_turns)
print("\n--- Generated code ---\n", code_str)

## 6. Execute the model and score it

`evaluate_code` runs the code **in isolation** (subprocess + timeout), serialises the
built plant and compares it to the reference. Three scores are computed:

- **Structure** – components (by type) and connections.
- **Measures** – simulated parameters (`V_liq`, `V_gas`, `T_ad`, …) within the acceptance band.
- **Missing values** – asked for or plausibly filled instead of silently invented.

We pass the LLM's questions as `response` – this feeds into the missing-values score.

In [ ]:
response = {"open_questions": questions or []}
report = evaluate_code(dp, code_str, response)

print(report.pretty())
print("\nScores:", {
    "build_success": report.build_success,
    "structure": round(report.structure, 3),
    "measures": round(report.measures, 3),
    "gaps": round(report.gaps, 3),
    "overall": report.overall(),
})

## 7. Plugging in another (free) LLM

Thanks to litellm you only change `MODEL` above (and possibly the matching API key) –
`call_llm` stays the same. A few options with free access:

| Provider | `MODEL` (example) | API key (env variable) |
| -------- | ----------------- | ---------------------- |
| Groq (free tier) | `groq/llama-3.3-70b-versatile` | `GROQ_API_KEY` |
| Groq vision (sketches) | `groq/meta-llama/llama-4-scout-17b-16e-instruct` | `GROQ_API_KEY` |
| Google Gemini (free tier) | `gemini/gemini-2.0-flash` | `GEMINI_API_KEY` |
| OpenRouter (free models) | `openrouter/<model>:free` | `OPENROUTER_API_KEY` |
| Ollama (local, no key) | `ollama/llama3` | – |

> Model names change; for the current list see the provider or the
> [litellm docs](https://docs.litellm.ai/docs/providers).

**For image/hybrid data points** (e.g. `BGA1_sketch`) pick a **vision model** – the
image wiring is already handled by `_to_litellm`.

**Evaluate the whole dataset:** for a run over all 24 data points there is the CLI
script `python benchmark/eval/solve.py` (see *Using the Dataset*).